<a href="https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/15_capstone_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- nav-header -->
[⬅ Previous](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/14_capstone_challenge.ipynb) · [🗺️ Course index](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb) · **Notebook 15 of the course**


# ✅ Notebook 15 — Capstone solutions

**Worked solutions for Notebook 14.** Try the challenge yourself first — the value is in the
attempt, not in reading the answer.

Everything below was really run on the same patients you used. **Every number on this page is
measured, not estimated.**

---

## The short version

| What we did | Held-out ROC-AUC |
|-------------|-----------------:|
| the example model from Notebook 14 | 0.754 |
| + many more features | 0.760 |
| + a gentler, more careful model | 0.760 |
| a random forest instead | 0.761 |
| **a plain logistic regression** | **0.768** |
| **all four models combined** | **0.777** |

So the best result is about **0.78**, starting from **0.75** — a gain of roughly **+0.02**.

Two things are worth noticing straight away:

- **The simplest model won among the single models.** Logistic regression beat XGBoost. Simple models
  are not automatically worse.
- **Combining models helped most**, and it needed no new ideas at all.

### But is that +0.02 real, or is it luck?

The last part of this notebook answers that, and it is the most useful thing here. In short, we ask
the same question three times, each time in a better way:

| How we ask | Answer |
|------------|--------|
| compare the two confidence intervals — **the common mistake** | they overlap → "no difference" |
| compare the models **on the same patients** (paired) | +0.023, twice as sharp, but still not conclusive |
| use **all 1,696 patients** instead of only 424 | **+0.026, clearly real (p < 0.001)** |

**The improvement was real the whole time.** Our first two ways of measuring it were simply too weak
to see it. That mistake is very common in published clinical research, and by the end of this
notebook you will have watched it happen and then be corrected.

**⏱️ Time:** 20 minutes to read · about 2 minutes to run.

## ⚙️ Run this cell first

In [1]:
# === ⚙️  Workshop setup — run this cell first ===============================
# Works in Google Colab and in local Jupyter. Installs anything missing, sets a
# clean plotting style, and gives you helpers to load the data.
# (This cell is identical in every notebook of the course.)
import importlib.util, subprocess, sys, os, random, warnings
warnings.filterwarnings("ignore")

# --- reproducibility: everyone in the room gets the same numbers ---------------
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)

# 📥 Where the workshop data comes from — already set up for you, nothing to do.
# The data downloads automatically the first time you need it. If you were given a
# different link, just paste it in place of the one below. These forms all work:
#   • a Google-Drive folder link      • a Drive / Dropbox / OneDrive file link
#   • a folder URL ending in "/"      • a link straight to a .zip
# Set it to "" if you would rather upload the CSVs by hand.
# (The data is not in the GitHub repo: it is real de-identified patient data covered
#  by a data use agreement and may not be redistributed openly.)
WORKSHOP_DATA_URL = os.environ.get(
    "WORKSHOP_DATA_URL",
    "https://drive.google.com/drive/folders/1y7CparhrqdlCAZq6xQlj8fZda394fniD")

def _ensure(pkgs):
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing])
_ensure({"numpy":"numpy","pandas":"pandas","sklearn":"scikit-learn",
         "matplotlib":"matplotlib","seaborn":"seaborn","shap":"shap","xgboost":"xgboost"})

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(RANDOM_STATE)          # seeds the legacy global np.random.* calls
RNG = np.random.default_rng(RANDOM_STATE)   # the modern generator — use this one
pd.set_option("display.max_columns", 120); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5); plt.rcParams["figure.dpi"] = 110

# Every model, split and resample in this course passes random_state=RANDOM_STATE, so
# your numbers should match your neighbour's exactly. (Different library *versions* can
# still shift the last decimal — that is normal and not a mistake on your part.)

# --- data loading: works locally AND remembers your upload across notebooks -----
_CACHE = {"dir": "unset"}   # memo so we only touch Google Drive once per session

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _drive_cache():
    """In Google Colab, mount Drive ONCE and return a persistent folder. A file you
    upload in one notebook is saved here, so every other notebook opens it automatically
    — no re-uploading. Returns None outside Colab, or if you decline to connect Drive."""
    if _CACHE["dir"] != "unset":
        return _CACHE["dir"]
    result = None
    if _in_colab():
        try:
            from google.colab import drive
            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive")
            result = "/content/drive/MyDrive/sepsis_workshop_data"
            os.makedirs(result, exist_ok=True)
        except Exception:
            result = None
    _CACHE["dir"] = result
    return result

def _find(name):
    """Look for the file on disk. Deliberately does NOT touch Google Drive, so the normal
    path never triggers an authorisation popup."""
    for p in [name, f"data/{name}", f"../data/{name}", f"workshop/data/{name}"]:
        if os.path.exists(p):
            return p
    return None

def _find_in_drive(name):
    """Only used as a fallback, because it mounts Drive (and that means a popup)."""
    cache = _drive_cache()
    if cache:
        p = os.path.join(cache, name)
        if os.path.exists(p):
            return p
    return None

def _direct_url(u):
    """Turn an ordinary Google-Drive / Dropbox / OneDrive *share* link into one that a
    plain HTTP client can actually download, so you can paste the link you were given."""
    import re
    m = (re.search(r"drive\.google\.com/file/d/([\w-]+)", u)
         or re.search(r"drive\.google\.com/(?:open|uc)\?(?:export=\w+&)?id=([\w-]+)", u))
    if m:
        return f"https://drive.google.com/uc?export=download&id={m.group(1)}"
    if "dropbox.com" in u:
        return u.split("?")[0] + "?dl=1"
    if "sharepoint.com" in u or "1drv.ms" in u:
        return u + ("&" if "?" in u else "?") + "download=1"
    return u

def _fetch(url, dest):
    import urllib.request, shutil as _sh
    req = urllib.request.Request(_direct_url(url), headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
        _sh.copyfileobj(r, f)

_FOLDER = {"done": False}

def _gdrive_folder(name):
    """WORKSHOP_DATA_URL points at a Google-Drive *folder*: fetch it once with gdown
    (a folder cannot be downloaded with a plain HTTP request), then serve files from it."""
    dest = "_workshop_data"
    if not _FOLDER["done"]:
        _ensure({"gdown": "gdown"})
        import gdown
        print("⬇  fetching the workshop data from Google Drive (just once) …")
        gdown.download_folder(url=WORKSHOP_DATA_URL, output=dest, quiet=True, use_cookies=False)
        _FOLDER["done"] = True
    for root, _dirs, files in os.walk(dest):
        if name in files:
            return os.path.join(root, name)
    return None

def _try_download(name):
    """Fetch the data from WORKSHOP_DATA_URL, if one was configured."""
    u = (WORKSHOP_DATA_URL or "").strip()
    if not u:
        return None
    try:
        if "/drive/folders/" in u:
            return _gdrive_folder(name)
        if u.lower().split("?")[0].endswith(".zip"):
            import zipfile
            bundle = "_workshop_data.zip"
            if not os.path.exists(bundle):
                print("⬇  downloading the workshop data bundle …")
                _fetch(u, bundle)
            with zipfile.ZipFile(bundle) as z:      # flatten any folder inside the zip
                for member in z.namelist():
                    if os.path.basename(member) == name:
                        with z.open(member) as src, open(name, "wb") as dst:
                            dst.write(src.read())
                        return name
            print(f"  ({name} was not inside the bundle)")
            return None
        print(f"⬇  downloading {name} …")
        _fetch(u.rstrip("/") + "/" + name, name)
        return name
    except Exception as e:
        print(f"  (download failed: {e})")
        for leftover in (name, "_workshop_data.zip"):
            if os.path.exists(leftover) and os.path.getsize(leftover) == 0:
                os.remove(leftover)
        return None

def _cache_to_drive(name, data):
    cache = _drive_cache()
    if cache:
        dest = os.path.join(cache, name)
        data.to_csv(dest, index=False)
        print(f"  💾 saved to Google Drive ({dest}) — no need to fetch it again.")

def load_csv(name):
    """Load a data CSV: looks on disk, then downloads it from WORKSHOP_DATA_URL, then checks
    your Google-Drive cache, and only as a last resort asks you to upload it — in which case
    it saves a copy to Drive so you never have to upload it twice."""
    p = _find(name)                       # 1. already on disk?
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    p = _try_download(name)               # 2. the built-in download link
    if p:
        print(f"✓ loaded {name}")
        return pd.read_csv(p)
    p = _find_in_drive(name)              # 3. a copy you saved on a previous run
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    try:                                  # 4. last resort: upload it by hand
        from google.colab import files
        print(f"⤴  Upload {name} just once — I'll save it so the other notebooks open it automatically:")
        up = files.upload()
        fname = list(up.keys())[0]
        data = pd.read_csv(fname)
        _cache_to_drive(name, data)
        return data
    except Exception:
        raise FileNotFoundError(
            f"Could not find {name}. Either paste your download link into WORKSHOP_DATA_URL at "
            f"the top of this cell, or put the CSV next to this notebook / in a data/ folder."
        )


## 📂 The same data and the same two groups

Identical to Notebook 14 — same seed, same patients — so the numbers can be compared directly.

In [2]:
def build_patient_table(ts):
    """Aggregate the 4-hourly time-series into ONE row per ICU stay.
       (This is exactly what Notebook 04 teaches you to build.)"""
    df = ts.copy()
    # clean temperature: prefer Celsius; repair obvious Fahrenheit-entry errors; drop impossible
    tf = df["Temp_F"].where((df["Temp_F"] >= 90) & (df["Temp_F"] <= 110))
    tc = df["Temp_C"].where((df["Temp_C"] >= 25) & (df["Temp_C"] <= 45))
    df["Temp_C_clean"] = tc.fillna((tf - 32) * 5/9)
    # impossible vitals -> NaN, then forward/back-fill WITHIN each stay
    for c, lo, hi in [("HR",20,300),("SysBP",40,300),("MeanBP",20,220),("RR",3,80),("SpO2",30,100)]:
        df[c] = df[c].where((df[c] >= lo) & (df[c] <= hi))
    g = df.groupby("icustayid", group_keys=False)
    vit = ["HR","SysBP","MeanBP","RR","SpO2","Temp_C_clean"]
    df[vit] = g[vit].apply(lambda x: x.ffill().bfill())
    # winsorise skewed labs so one stray value can't define a min/max
    for c in ["Arterial_lactate","Creatinine","BUN","WBC_count","Platelets_count","INR","Glucose"]:
        lo, hi = df[c].quantile(0.01), df[c].quantile(0.99)
        df[c] = df[c].clip(lo, hi)
    tv = ["HR","SysBP","MeanBP","RR","SpO2","Temp_C_clean","GCS","Creatinine","BUN",
          "Arterial_lactate","WBC_count","Platelets_count","Potassium","Sodium",
          "Albumin","INR","Arterial_pH","Shock_Index","SOFA","SIRS","PaO2_FiO2","Hb"]
    grp = df.groupby("icustayid")
    f = {}
    f["age"]=grp["age"].first(); f["gender"]=grp["gender"].first()
    f["elixhauser"]=grp["elixhauser"].first(); f["re_admission"]=grp["re_admission"].first().astype(int)
    f["weight_kg"]=grp["Weight_kg"].median(); f["n_blocs"]=grp.size(); f["los_hours"]=grp["bloc"].max()*4
    f["mechvent_ever"]=grp["mechvent"].max(); f["vaso_max"]=grp["max_dose_vaso"].max()
    f["fluid_balance_last"]=grp["cumulated_balance"].last(); f["urine_total"]=grp["output_step"].sum()
    for c in tv:
        f[f"{c}_mean"]=grp[c].mean(); f[f"{c}_min"]=grp[c].min()
        f[f"{c}_max"]=grp[c].max();  f[f"{c}_last"]=grp[c].last()
    for c in ["Creatinine","Arterial_lactate","SOFA","Shock_Index","GCS"]:
        f[f"{c}_delta"]=grp[c].last()-grp[c].first()
    X = pd.DataFrame(f); X["morta_90"]=grp["morta_90"].max(); X["died_in_hosp"]=grp["died_in_hosp"].max()
    return X.reset_index()

def load_patients():
    p = _find("sepsis_patients.csv")
    if p:
        print(f"✓ loaded {p}"); return pd.read_csv(p)
    print("sepsis_patients.csv not found — building it from the time-series file…")
    return build_patient_table(load_csv("sepsis_timeseries.csv"))

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ts = load_csv("sepsis_timeseries.csv")

stay_outcome = ts.groupby("icustayid")["morta_90"].max()
TRAIN_IDS, TEST_IDS = train_test_split(
    stay_outcome.index.to_numpy(), test_size=0.25,
    stratify=stay_outcome.to_numpy(), random_state=2026)
TRAIN_IDS, TEST_IDS = set(TRAIN_IDS.tolist()), set(TEST_IDS.tolist())

ts_train = ts[ts["icustayid"].isin(TRAIN_IDS)].copy()
ts_test  = ts[ts["icustayid"].isin(TEST_IDS)].copy()
Y_TEST   = ts_test.groupby("icustayid")["morta_90"].max()
ytr      = ts_train.groupby("icustayid")["morta_90"].max()
SPW      = float((ytr == 0).sum() / max(1, (ytr == 1).sum()))
cv       = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

_rng = np.random.default_rng(RANDOM_STATE)

def auc_ci(y, p, n_boot=400):
    """95% confidence interval of the ROC-AUC for ONE model."""
    y, p = np.asarray(y), np.asarray(p)
    a = []
    for _ in range(n_boot):
        i = _rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) > 1:
            a.append(roc_auc_score(y[i], p[i]))
    return float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))

RESULTS = []

def report(name, p, cv_score=None):
    """Measure one set of predictions and remember the result."""
    p = pd.Series(p).reindex(Y_TEST.index)
    lo, hi = auc_ci(Y_TEST, p)
    RESULTS.append({"model": name, "CV": cv_score, "test_AUC": roc_auc_score(Y_TEST, p),
                    "CI_low": lo, "CI_high": hi,
                    "avg_precision": average_precision_score(Y_TEST, p),
                    "Brier": brier_score_loss(Y_TEST, p)})
    r = RESULTS[-1]
    cvtxt = f"CV {cv_score:.4f} · " if cv_score is not None else ""
    print(f"{name:36} {cvtxt}held-out {r['test_AUC']:.4f} [{lo:.3f}-{hi:.3f}]")
    return p

print(f"training patients {len(TRAIN_IDS):,} · held-out patients {len(TEST_IDS):,}")

✓ loaded data/sepsis_timeseries.csv
training patients 1,272 · held-out patients 424


---

## Solution 1 — many more features

This is where most of the honest improvement comes from, and it is the skill from Notebook 04.

The example model only used **summary values** for each patient: mean, minimum, maximum, last. Those
throw away the *shape* of the stay. We add five kinds of information:

1. **Variability** — the standard deviation of each measurement. A patient whose blood pressure swings
   widely is not the same as a stable patient with the same average.
2. **Direction of travel** — how much a value changed from first to last, divided by the length of the
   stay. A lactate that is rising is worse than one that is falling.
3. **The last 24 hours** — the mean, and the change, over the final 6 blocks. What happened recently
   matters more than what happened on day one.
4. **Missing values as information** — how often a laboratory test was *not* done. If the team keeps
   ordering blood gases, they are worried. The absence of a test measures clinical concern.
5. **How much treatment was given** — hours on vasopressors, hours ventilated, total fluid.

That takes us from 104 features to 185.

In [4]:
VITALS = ["HR", "MeanBP", "SysBP", "RR", "SpO2", "Arterial_lactate", "SOFA",
          "Creatinine", "BUN", "GCS", "WBC_count", "Platelets_count", "Arterial_pH",
          "Shock_Index", "PaO2_FiO2", "INR", "Albumin"]

def basic_features(sl):
    """The features the example model in Notebook 14 used."""
    t = build_patient_table(sl).set_index("icustayid")
    return t.drop(columns=["morta_90", "died_in_hosp"])

def rich_features(sl):
    """The basic features plus everything described above."""
    t = basic_features(sl)
    sl = sl.sort_values(["icustayid", "bloc"])
    g = sl.groupby("icustayid")
    n = g.size()

    # 1. variability - how much did the value move around?
    for c in VITALS:
        t[f"{c}_std"] = g[c].std()

    # 2. direction of travel, per block so long and short stays are comparable
    for c in VITALS:
        t[f"{c}_rate"] = (g[c].last() - g[c].first()) / n.clip(lower=1)

    # 3. the last 24 hours (6 blocks of 4 hours)
    mx  = sl.groupby("icustayid")["bloc"].transform("max")
    l24 = sl[sl["bloc"] > mx - 6]
    g24 = l24.groupby("icustayid")
    for c in VITALS:
        t[f"{c}_d24"]    = g24[c].last() - g24[c].first()
        t[f"{c}_mean24"] = g24[c].mean()

    # 4. missing values carry information about clinical concern
    for c in ["Arterial_lactate", "Albumin", "INR", "Arterial_pH", "PaO2_FiO2", "Total_bili"]:
        t[f"{c}_miss"] = g[c].apply(lambda s: s.isna().mean())

    # 5. how much treatment the patient received
    t["vaso_hours"]     = g["max_dose_vaso"].apply(lambda s: (s > 0).sum()) * 4
    t["vaso_auc"]       = g["max_dose_vaso"].sum()
    t["mechvent_hours"] = g["mechvent"].sum() * 4
    t["fluid_in_total"] = g["input_step"].sum()
    t["balance_rate"]   = g["cumulated_balance"].last() / n.clip(lower=1)

    # the worst 24-hour window of the whole stay
    for c in ["SOFA", "Arterial_lactate"]:
        t[f"{c}_roll24max"] = (sl.groupby("icustayid")[c]
                                 .rolling(6, min_periods=1).mean()
                                 .groupby("icustayid").max().values)
    return t

Xtr_b = basic_features(ts_train)
Xte_b = basic_features(ts_test).reindex(columns=Xtr_b.columns)
Xtr_r = rich_features(ts_train)
Xte_r = rich_features(ts_test).reindex(columns=Xtr_r.columns)
y     = ytr.reindex(Xtr_r.index)

print(f"basic features: {Xtr_b.shape[1]}")
print(f"rich  features: {Xtr_r.shape[1]}")

basic features: 104
rich  features: 185


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

def make_xgb(**kw):
    d = dict(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
             colsample_bytree=0.8, eval_metric="logloss", scale_pos_weight=SPW,
             random_state=RANDOM_STATE, n_jobs=-1)
    d.update(kw)
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("model", XGBClassifier(**d))])

def fit_and_report(name, model, Xtr, Xte):
    c = cross_val_score(model, Xtr, y, cv=cv, scoring="roc_auc", n_jobs=-1).mean()
    model.fit(Xtr, y)
    p = pd.Series(model.predict_proba(Xte)[:, 1], index=Xte.index)
    return report(name, p, c)

p_basic = fit_and_report("example model (basic features)", make_xgb(), Xtr_b, Xte_b)
p_rich  = fit_and_report("same model, rich features",      make_xgb(), Xtr_r, Xte_r)

example model (basic features)       CV 0.7788 · held-out 0.7540 [0.692-0.817]


same model, rich features            CV 0.7874 · held-out 0.7600 [0.701-0.814]


**What happened.** Cross-validation improved clearly (about 0.779 → 0.787) and the held-out score
improved a little (0.754 → 0.760).

The *cross-validation* improvement is bigger than the *held-out* improvement. That is normal and worth
understanding: cross-validation averages over ~1,270 patients, so it is a steadier measurement. The
held-out score uses only 424 patients, so it moves around more. Hold on to that thought — it becomes
the main lesson at the end.

---

## Solution 2 — a gentler model

With 185 features and only ~1,270 patients, a model can easily memorise the training data instead of
learning from it. So we make it more careful:

- **shallower trees** (`max_depth` 4 → 3): each tree may ask fewer questions in a row,
- **smaller steps** (`learning_rate` 0.05 → 0.03) with more trees to compensate,
- **each tree sees fewer features** (`colsample_bytree` 0.8 → 0.6),
- **a leaf must contain more patients** (`min_child_weight` 5) and a **penalty on complexity**
  (`reg_lambda` 3).

None of this is magic. Every item says the same thing: *be less confident*.

In [6]:
p_tuned = fit_and_report(
    "rich features + gentler model",
    make_xgb(n_estimators=700, max_depth=3, learning_rate=0.03,
             colsample_bytree=0.6, min_child_weight=5, reg_lambda=3.0),
    Xtr_r, Xte_r)

rich features + gentler model        CV 0.7973 · held-out 0.7604 [0.708-0.807]


Cross-validation rises again (to about 0.797); the held-out score stays at about 0.760.

---

## Solution 3 — try other kinds of model

Gradient boosting is a good default, but it is not always the winner. Here are three alternatives on
exactly the same features.

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

p_rf = fit_and_report("random forest", Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(n_estimators=600, min_samples_leaf=3,
                                     class_weight="balanced", n_jobs=-1,
                                     random_state=RANDOM_STATE))]), Xtr_r, Xte_r)

# Logistic regression needs the features on a common scale, and C=0.05 keeps the
# coefficients small so 185 features cannot overwhelm ~1,270 patients.
p_lr = fit_and_report("logistic regression", Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
    ("model",  LogisticRegression(max_iter=4000, C=0.05, class_weight="balanced",
                                  random_state=RANDOM_STATE))]), Xtr_r, Xte_r)

p_hgb = fit_and_report("hist gradient boosting", Pipeline([
    ("model", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.04, max_iter=400,
                                             l2_regularization=1.0,
                                             random_state=RANDOM_STATE))]), Xtr_r, Xte_r)

random forest                        CV 0.7942 · held-out 0.7608 [0.699-0.819]


logistic regression                  CV 0.7837 · held-out 0.7678 [0.704-0.822]


Exception in thread Thread-58 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\lolka\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\lolka\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\lolka\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "C:\Users\lolka\AppData\Local\Programs\Python\Python312\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 120: character maps to <undefined>


hist gradient boosting               CV 0.7892 · held-out 0.7555 [0.695-0.812]


### 👀 The surprise

**Logistic regression gives the best held-out score of all the single models (about 0.768)** — better
than XGBoost, better than the random forest.

Logistic regression is the oldest and simplest method here: essentially a weighted sum of the
features. It is also the one you can print in a paper, hand to a clinician, and defend line by line.

Two things to take from this:

1. **Always include a simple model in your comparison.** If you had only tried gradient boosting, you
   would have concluded that 0.760 was the best available — and you would have been wrong.
2. **Do not over-read it either.** The gap between 0.768 and 0.760 is small. By the end of this
   notebook you will be able to say precisely how much confidence such a gap deserves.

---

## Solution 4 — combine the models (this helped most)

Different models make different mistakes. If you average them, the mistakes partly cancel out while
the shared signal remains.

We average the **ranks** rather than the probabilities. A rank means "this patient is the 12th
riskiest". This avoids the problem that logistic regression and XGBoost produce probabilities on
different scales — a fair average needs a common unit.

In [8]:
ranked = [pd.Series(p).rank(pct=True) for p in [p_tuned, p_rf, p_lr, p_hgb]]
p_ensemble = sum(ranked) / len(ranked)

report("ENSEMBLE of the four models", p_ensemble)
pd.DataFrame(RESULTS).round(4)

ENSEMBLE of the four models          held-out 0.7766 [0.721-0.830]


,model,CV,test_AUC,CI_low,CI_high,avg_precision,Brier
0,example model (basic features),0.7788,0.7540,0.6924,0.8167,0.3966,0.1462
1,"same model, rich features",0.7874,0.7600,0.7009,0.8137,0.4000,0.1419
2,rich features + gentler model,0.7973,0.7604,0.7080,0.8073,0.3881,0.1456
3,random forest,0.7942,0.7608,0.6995,0.8187,0.4075,0.1455
4,logistic regression,0.7837,0.7678,0.7039,0.8218,0.4352,0.1782
5,hist gradient boosting,0.7892,0.7555,0.6953,0.8117,0.4077,0.1350
6,ENSEMBLE of the four models,NaN,0.7766,0.7207,0.8299,0.4253,0.2455


**About 0.777 — the best result so far**, and it came from combining models we already had, not
from a new idea.

This is why ensembles are everywhere in competitions. It is also why they are less common in
hospitals: you now have four models to validate, monitor and explain instead of one. That is a real
cost, and it belongs in the decision.

> ⚠️ **Look at the ensemble's Brier score in the table — it is terrible (about 0.25).** That is not a
> bug, and it is worth understanding. We averaged **ranks**, so the output is a position in a queue,
> not a probability. It is spread evenly between 0 and 1, so a patient at rank 0.9 gets "90% risk"
> even though only 18% of patients die. The *order* is excellent; the *numbers* are meaningless.
>
> This is exactly the difference between ROC-AUC and the Brier score. If you needed real
> probabilities, you would put the calibration step from Solution 5 on top of the ensemble.

---

## Solution 5 — believable probabilities (calibration)

ROC-AUC only asks whether patients are in the right **order**. It does not care whether the numbers
are believable. But if a model tells a doctor "20% risk", then about 20 out of 100 such patients
should really die — otherwise the number cannot support a decision.

**Calibration** repairs the numbers without changing the order. We measure it with the **Brier
score**, where **lower is better**.

In [9]:
from sklearn.calibration import CalibratedClassifierCV

calibrated = CalibratedClassifierCV(
    make_xgb(n_estimators=700, max_depth=3, learning_rate=0.03,
             colsample_bytree=0.6, min_child_weight=5, reg_lambda=3.0),
    method="isotonic", cv=5)
calibrated.fit(Xtr_r, y)
p_cal = pd.Series(calibrated.predict_proba(Xte_r)[:, 1], index=Xte_r.index)

before = next(r for r in RESULTS if r["model"] == "rich features + gentler model")
report("gentler model + calibration", p_cal)
after = RESULTS[-1]
print(f"\nBrier score  before {before['Brier']:.4f}  ->  after {after['Brier']:.4f}   (lower is better)")
print(f"ROC-AUC      before {before['test_AUC']:.4f}  ->  after {after['test_AUC']:.4f}   (barely moves)")

gentler model + calibration          held-out 0.7607 [0.697-0.816]

Brier score  before 0.1456  ->  after 0.1323   (lower is better)
ROC-AUC      before 0.7604  ->  after 0.7607   (barely moves)


Exactly as expected: the Brier score improves, the ROC-AUC hardly changes.

**If you are choosing a model to actually use, this matters more than a small AUC difference.**

---

## Solution 6 — the early-warning version

Everything so far used the patient's **whole** ICU stay. That is fine for a study, but useless at the
bedside: by the time the stay is over, the decisions have been made.

So we repeat the experiment using **only the first 24 hours** of each stay.

In [10]:
early_train = ts_train[ts_train["bloc"] <= 6]
early_test  = ts_test[ts_test["bloc"] <= 6]

Xtr_e = rich_features(early_train)
Xte_e = rich_features(early_test).reindex(columns=Xtr_e.columns)
y_e   = ytr.reindex(Xtr_e.index)

early = make_xgb(n_estimators=700, max_depth=3, learning_rate=0.03,
                 colsample_bytree=0.6, min_child_weight=5, reg_lambda=3.0)
early.fit(Xtr_e, y_e)
p_early = pd.Series(early.predict_proba(Xte_e)[:, 1], index=Xte_e.index)

report("first 24 hours only", p_early)
full = next(r for r in RESULTS if r["model"] == "rich features + gentler model")
print(f"\nWhole stay : {full['test_AUC']:.4f}")
print(f"First 24 h : {RESULTS[-1]['test_AUC']:.4f}")
print(f"Difference : {RESULTS[-1]['test_AUC'] - full['test_AUC']:+.4f}")

first 24 hours only                  held-out 0.7074 [0.645-0.767]

Whole stay : 0.7604
First 24 h : 0.7074
Difference : -0.0531


The score drops by roughly **0.05**. Nothing went wrong — this is the honest cost of predicting
**early enough to be useful**.

It is also the number an editor or a clinical director should ask for. A model that only performs
well once the stay is finished has, in a sense, been told the answer by the passage of time.

---

## The results so far

In [11]:
final = pd.DataFrame(RESULTS).sort_values("test_AUC", ascending=False).reset_index(drop=True)
final["CI"] = final.apply(lambda r: f"{r['CI_low']:.3f}-{r['CI_high']:.3f}", axis=1)
display(final[["model", "CV", "test_AUC", "CI", "avg_precision", "Brier"]].round(4))

,model,CV,test_AUC,CI,avg_precision,Brier
0,ENSEMBLE of the four models,NaN,0.7766,0.721-0.830,0.4253,0.2455
1,logistic regression,0.7837,0.7678,0.704-0.822,0.4352,0.1782
2,random forest,0.7942,0.7608,0.699-0.819,0.4075,0.1455
3,gentler model + calibration,NaN,0.7607,0.697-0.816,0.4074,0.1323
4,rich features + gentler model,0.7973,0.7604,0.708-0.807,0.3881,0.1456
5,"same model, rich features",0.7874,0.7600,0.701-0.814,0.4000,0.1419
6,hist gradient boosting,0.7892,0.7555,0.695-0.812,0.4077,0.1350
7,example model (basic features),0.7788,0.7540,0.692-0.817,0.3966,0.1462
8,first 24 hours only,NaN,0.7074,0.645-0.767,0.3259,0.1625


---

# 🔬 The final question: was the improvement real?

We went from 0.754 to 0.777 on the held-out patients — a gain of about **+0.023**. Before writing
that in a paper, you must show it is not simply luck.

We will now ask that question **three times, each time in a better way**. Watch the answer change.

### Step 1 — the common mistake: comparing two confidence intervals

Almost everybody does this: print a confidence interval for each model, notice that they overlap, and
conclude "the difference is not significant".

In [12]:
e = next(r for r in RESULTS if r["model"] == "example model (basic features)")
n = next(r for r in RESULTS if r["model"] == "ENSEMBLE of the four models")

print(f"example model : {e['test_AUC']:.4f}   95% CI [{e['CI_low']:.3f}, {e['CI_high']:.3f}]")
print(f"ensemble      : {n['test_AUC']:.4f}   95% CI [{n['CI_low']:.3f}, {n['CI_high']:.3f}]")
print(f"\nDo the intervals overlap? {'YES' if e['CI_high'] > n['CI_low'] else 'no'}")
print("Usual (and wrong) conclusion: 'the difference is not significant'.")

example model : 0.7540   95% CI [0.692, 0.817]
ensemble      : 0.7766   95% CI [0.721, 0.830]

Do the intervals overlap? YES
Usual (and wrong) conclusion: 'the difference is not significant'.


**That reasoning is wrong, and it is wrong in a way that matters.**

Each interval answers the question *"how much would this model's score move if I had a different
sample of patients?"*. But we do **not** have two different samples. Both models were measured on
**exactly the same 424 patients**. If that group happened to contain a few unusually difficult
patients, *both* scores went down together.

Comparing two separate intervals throws that shared information away. We can do much better.

### Step 2 — the correct comparison: same patients, look at the difference

Now we resample patients, and each time we score **both models on that same resample** and record the
**difference**. Because the two models rise and fall together, the difference is far more stable than
either score on its own.

In [13]:
def paired_difference(y, p_a, p_b, n_boot=2000, seed=0):
    """Bootstrap the DIFFERENCE between two models measured on the same patients."""
    r = np.random.default_rng(seed)
    y = np.asarray(y); a = np.asarray(p_a); b = np.asarray(p_b)
    d = []
    for _ in range(n_boot):
        i = r.integers(0, len(y), len(y))
        if len(np.unique(y[i])) > 1:
            d.append(roc_auc_score(y[i], b[i]) - roc_auc_score(y[i], a[i]))
    d = np.array(d)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), (d <= 0).mean()

m, lo, hi, p = paired_difference(Y_TEST, p_basic.reindex(Y_TEST.index),
                                 p_ensemble.reindex(Y_TEST.index))
print(f"paired difference (ensemble - example) : {m:+.4f}")
print(f"95% confidence interval                : [{lo:+.4f}, {hi:+.4f}]")
print(f"probability the ensemble is NOT better : {p:.3f}")
print()
print(f"width of the interval on the DIFFERENCE : {hi - lo:.3f}")
print(f"width of one model's own interval       : {e['CI_high'] - e['CI_low']:.3f}   <- about twice as wide")

paired difference (ensemble - example) : +0.0226
95% confidence interval                : [-0.0057, +0.0520]
probability the ensemble is NOT better : 0.060

width of the interval on the DIFFERENCE : 0.058
width of one model's own interval       : 0.124   <- about twice as wide


The interval on the **difference** is roughly **half as wide** as the interval on either score.
Same data, same models — we simply asked the question properly.

But it still includes zero, only just. With 424 patients, of whom 77 died, we cannot yet prove a
+0.02 improvement. **We are not wrong — we are under-powered.**

### Step 3 — stop throwing away 75% of the data

Here is the real problem. To run a fair competition we set **1,272 patients aside** and measured on
only 424. That is the right thing to do for a leaderboard, where everyone must be judged on identical
patients. It is a wasteful way to answer a *scientific* question.

There is a better way: **out-of-fold prediction**. Split all 1,696 patients into 5 parts. Train on 4
parts, predict the 5th. Repeat until every patient has been predicted exactly once by a model that
never saw them. Now every patient contributes to the answer, and no patient influenced its own
prediction.

We repeat the whole procedure with 3 different splits, to check the result is not an accident of one
particular division.

*(This cell trains 75 models and takes about 40 seconds.)*

In [14]:
X_all_basic = basic_features(ts)
X_all_rich  = rich_features(ts).reindex(X_all_basic.index)
y_all       = ts.groupby("icustayid")["morta_90"].max().reindex(X_all_basic.index)

def ensemble_members():
    return [make_xgb(n_estimators=700, max_depth=3, learning_rate=0.03, colsample_bytree=0.6,
                     min_child_weight=5, reg_lambda=3.0),
            Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("model", RandomForestClassifier(n_estimators=600, min_samples_leaf=3,
                                                       class_weight="balanced", n_jobs=-1,
                                                       random_state=RANDOM_STATE))]),
            Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
                      ("model", LogisticRegression(max_iter=4000, C=0.05, class_weight="balanced",
                                                   random_state=RANDOM_STATE))]),
            Pipeline([("model", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.04,
                                                               max_iter=400, l2_regularization=1.0,
                                                               random_state=RANDOM_STATE))])]

def out_of_fold(seed):
    """Predict every patient with a model that never saw that patient."""
    folds = StratifiedKFold(5, shuffle=True, random_state=seed)
    pe = pd.Series(np.nan, index=X_all_basic.index)   # example model
    pn = pd.Series(np.nan, index=X_all_basic.index)   # ensemble
    for tr, te in folds.split(X_all_basic, y_all):
        m = make_xgb(); m.fit(X_all_basic.iloc[tr], y_all.iloc[tr])
        pe.iloc[te] = m.predict_proba(X_all_basic.iloc[te])[:, 1]
        ranks = []
        for mm in ensemble_members():
            mm.fit(X_all_rich.iloc[tr], y_all.iloc[tr])
            ranks.append(pd.Series(mm.predict_proba(X_all_rich.iloc[te])[:, 1]).rank(pct=True).to_numpy())
        pn.iloc[te] = np.mean(ranks, axis=0)
    return pe, pn

runs, all_e, all_n = [], [], []
for seed in [11, 22, 33]:
    pe, pn = out_of_fold(seed)
    all_e.append(pe); all_n.append(pn)
    a, b = roc_auc_score(y_all, pe), roc_auc_score(y_all, pn)
    runs.append({"split seed": seed, "example": a, "ensemble": b, "difference": b - a})
    print(f"  split {seed}: example {a:.4f}   ensemble {b:.4f}   difference {b - a:+.4f}", flush=True)

runs = pd.DataFrame(runs)
print(f"\nmean difference over the 3 splits : {runs['difference'].mean():+.4f}")
print(f"does every split favour the ensemble? {bool((runs['difference'] > 0).all())}")

  split 11: example 0.7794   ensemble 0.8087   difference +0.0293


  split 22: example 0.7861   ensemble 0.8154   difference +0.0292


  split 33: example 0.7717   ensemble 0.8026   difference +0.0309



mean difference over the 3 splits : +0.0298
does every split favour the ensemble? True


In [15]:
# average the three runs, then do the paired test on all 1,696 patients
pe_all = sum(all_e) / len(all_e)
pn_all = sum(all_n) / len(all_n)

m2, lo2, hi2, p2 = paired_difference(y_all, pe_all, pn_all)
print(f"Using all {len(y_all):,} patients ({int(y_all.sum())} of them died):\n")
print(f"  example model : {roc_auc_score(y_all, pe_all):.4f}")
print(f"  ensemble      : {roc_auc_score(y_all, pn_all):.4f}")
print(f"  difference    : {m2:+.4f}   95% CI [{lo2:+.4f}, {hi2:+.4f}]")
print(f"  probability the ensemble is NOT better: {p2:.4f}")
print()
print("VERDICT:", "the improvement is REAL" if lo2 > 0 else "still not proven")

Using all 1,696 patients (309 of them died):

  example model : 0.7880
  ensemble      : 0.8138
  difference    : +0.0257   95% CI [+0.0145, +0.0373]
  probability the ensemble is NOT better: 0.0000

VERDICT: the improvement is REAL


### 🎯 The result

Now the answer is unambiguous. The confidence interval on the difference lies **entirely above zero**,
all three splits agree, and the probability of seeing this by chance is below 1 in 1,000.

**The improvement was real all along.** Our first two measurements were simply too weak to detect it.

Notice something else. On all 1,696 patients, *both* models score **higher** than on the 424 held-out
patients (about 0.79 and 0.81, instead of 0.754 and 0.777). Each fold trains on 1,357 patients instead
of 1,272, and more training data gives a better model. The competition score was pessimistic — exactly
as you would expect.

---

## 🧠 What to take away

**1. Never compare two models by asking whether their confidence intervals overlap.**
It is one of the most common statistical mistakes in prediction papers, and it is biased towards
concluding "no difference". Compare the models **on the same patients** and put the confidence
interval on the **difference**. Here that alone made the interval twice as sharp.

**2. "Not significant" usually means "not enough patients", not "no effect".**
The improvement did not appear or disappear between step 1 and step 3. Only our ability to see it
changed. Before concluding that something does not work, ask whether your study could have detected
it if it did.

**3. A single held-out group is for a fair contest, not for a precise answer.**
It is the right choice for a leaderboard, because everyone must be judged on identical patients. It is
the wrong choice for measuring a small difference, because you throw away most of your data. For the
scientific question, use out-of-fold prediction over everything you have.

**4. The win is still small.**
+0.026 is real and worth having, but it will not transform patient care on its own. That is why the
final questions in Notebook 14 — external validation, what action follows an alert, calibration,
fairness — matter more than the last decimal place.

> 🩺 **The honest summary:** better features and combining models moved this task from about 0.79 to
> about 0.81, and that improvement is statistically solid. Turning it into something that helps a
> patient still needs a prospective study, a clinical workflow, and a team.

## What we did *not* do

- **We did not tune hyperparameters systematically.** A `RandomizedSearchCV` might add a little.
  Based on Notebook 09, expect roughly +0.005 — far less than better features gave us.
- **We did not use the time series directly.** A sequence model (for example an LSTM) could read the
  blocks in order instead of summarising them. On tabular ICU data of this size it usually does not
  beat the approach here, and it is much harder to explain.
- **We did not add outside information.** Comorbidities, admission diagnosis, previous admissions and
  medication history would very likely help more than any model change on this page.

<!-- nav-footer -->
---

### 🎓 That's the end of the course

<a href="https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Back to the course index" height="32"/></a>

👉 **[Back to the course index](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb)** — every notebook stays open to you.

[⬅ Back to Notebook 14](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/14_capstone_challenge.ipynb) · [🗺️ Course index](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb) · [📁 The course on GitHub](https://github.com/lorenzkap/ML2026)